<a href="https://colab.research.google.com/github/Timmythaw/langgraph-adk-edu-comparison/blob/main/notebooks/05_hallucination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Notebook 05 — Hallucination Rate Analysis

> **Fully self-contained.** No dependency on notebooks 01–04.
> Builds both systems, runs all 3 scenarios, measures hallucination
> with DeepEval + a composite formula.

## Hallucination Rate Formula

$$\text{Hallucination Rate} = \frac{\text{Hallucinated Outputs}}{\text{Total Outputs}} \times 100$$

We measure this at **three complementary levels**:

| Level | Tool | What it detects |
|---|---|---|
| **Claim-level** | DeepEval `HallucinationMetric` | Claims not supported by retrieved context |
| **Retrieval-level** | RAGAS Faithfulness (imported from NB04 CSV or re-computed) | Output grounded in retrieved chunks? |
| **Factual-level** | LLM Judge accuracy score (from NB03 CSV or re-computed) | Factually correct content? |

**Composite Hallucination Rate:**
$$\text{Composite} = \frac{(1 - \text{Faithfulness}) + \text{DeepEval Score} + (1 - \frac{\text{Accuracy}}{5})}{3} \times 100$$

**Colab Secrets required:**
- `GOOGLE_CLOUD_PROJECT`
- `GOOGLE_CLOUD_LOCATION`
- `VERTEX_AI_SEARCH_DATASTORE_ID`
- `LANGSMITH_API_KEY`

| Section | What happens |
|---|---|
| 1 | Install dependencies |
| 2 | Secrets, auth, config |
| 3 | Context-saving Vertex AI Search tool |
| 4 | Build LangGraph system |
| 5 | Build Google ADK system |
| 6 | Runners |
| 7 | Run scenarios + capture contexts |
| 8 | DeepEval hallucination measurement |
| 9 | Load prior RAGAS + Judge scores (optional) |
| 10 | Composite hallucination rate |
| 11 | Visualisations |
| 12 | Export results |

## 1 — Install Dependencies

In [ ]:
!pip install deepeval ragas langgraph langchain-google-genai langchain-google-community \
    google-adk google-cloud-aiplatform google-cloud-discoveryengine \
    google-api-python-client google-auth-httplib2 google-auth-oauthlib \
    langsmith datasets -q
print("All packages installed.")

## 2 — Secrets, Auth & Config

In [ ]:
import os
from google.colab import userdata, auth

PROJECT_ID   = userdata.get("GOOGLE_CLOUD_PROJECT")
LOCATION     = userdata.get("GOOGLE_CLOUD_LOCATION")
DATASTORE_ID = userdata.get("VERTEX_AI_SEARCH_DATASTORE_ID")

os.environ["GOOGLE_CLOUD_PROJECT"]      = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"]     = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["LANGCHAIN_TRACING_V2"]      = "true"
os.environ["LANGCHAIN_API_KEY"]         = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"]         = "langgraph-adk-edu-comparison"
os.environ["LANGCHAIN_ENDPOINT"]        = "https://api.smith.langchain.com"

# DeepEval uses OPENAI_API_KEY by default but supports custom LLMs
# We override it with a Gemini wrapper in Section 8
os.environ["OPENAI_API_KEY"] = "placeholder-not-used"

auth.authenticate_user()

import vertexai
from google.cloud import aiplatform
vertexai.init(project=PROJECT_ID, location="us-central1")
aiplatform.init(project=PROJECT_ID, location="us-central1")

print("Project  :", PROJECT_ID[:20])
print("Location :", LOCATION)
print("Datastore:", DATASTORE_ID[:20])

## 3 — Context-Saving Vertex AI Search Tool

> Same hook as NB04 — every retrieved chunk is stored in `CONTEXT_STORE`
> keyed by `run_key` for use as DeepEval `context`.

In [ ]:
from google.cloud import discoveryengine_v1 as discoveryengine
from collections import defaultdict
import threading

CONTEXT_STORE: dict[str, list[str]] = defaultdict(list)
_current_run_key = threading.local()

def set_run_key(key: str):
    _current_run_key.value = key
    CONTEXT_STORE[key] = []

def get_run_key() -> str:
    return getattr(_current_run_key, "value", "__default__")

def retrieve_course_materials(query: str, page_size: int = 5) -> str:
    """Search curriculum datastore for relevant course materials.

    Automatically stores retrieved chunks in CONTEXT_STORE for hallucination evaluation.

    Args:
        query: Natural language search query.
        page_size: Number of results to retrieve.

    Returns:
        Concatenated snippet text, or 'No relevant materials found.'
    """
    client = discoveryengine.SearchServiceClient()
    serving_config = (
        f"projects/{PROJECT_ID}/locations/{LOCATION}"
        f"/collections/default_collection/dataStores/{DATASTORE_ID}"
        f"/servingConfigs/default_config"
    )
    request = discoveryengine.SearchRequest(
        serving_config=serving_config, query=query, page_size=page_size,
        content_search_spec=discoveryengine.SearchRequest.ContentSearchSpec(
            snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec.SnippetSpec(
                return_snippet=True
            ),
            summary_spec=discoveryengine.SearchRequest.ContentSearchSpec.SummarySpec(
                summary_result_count=min(page_size, 5), include_citations=True,
            ),
        ),
    )
    response = client.search(request)
    snippets = []
    for result in response.results:
        doc = result.document
        if doc.derived_struct_data:
            for s in doc.derived_struct_data.get("snippets", []):
                text = s.get("snippet", "").strip()
                if text:
                    snippets.append(text)

    # ── Context-saving hook ──────────────────────────────────────
    CONTEXT_STORE[get_run_key()].extend(snippets)
    # ─────────────────────────────────────────────────────────────

    return "\n\n---\n\n".join(snippets) if snippets else "No relevant materials found."

print("Context-saving Vertex AI Search tool ready.")

## 4 — Build LangGraph System

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage
from typing import TypedDict, Annotated, Literal
import operator

class TeacherState(TypedDict):
    messages:         Annotated[list, operator.add]
    task_type:        str
    course_materials: str
    draft_output:     str
    final_output:     str
    hitl_decision:    str

lg_orchestrator_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro", vertexai=True, project=PROJECT_ID,
    location="us-central1", max_output_tokens=4096, stream_usage=True,
    model_kwargs={"thinking": {"thinking_budget": 1024}},
)
lg_worker_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", vertexai=True, project=PROJECT_ID,
    location="us-central1", max_output_tokens=4096, stream_usage=True,
    model_kwargs={"thinking": {"thinking_budget": 512}},
)
print("LangGraph models ready.")

In [ ]:
def router_node(state: TeacherState) -> TeacherState:
    user_msg = state["messages"][-1].content
    lower = user_msg.lower()
    if any(k in lower for k in ["email", "send", "announcement", "draft"]):
        task = "email"
    elif any(k in lower for k in ["lesson plan", "outline", "lecture"]):
        task = "lessonplan"
    elif any(k in lower for k in ["quiz", "multiple choice", "question", "mcq"]):
        task = "quiz"
    else:
        prompt = (
            "Classify into exactly one of: lessonplan / quiz / email.\n"
            f"Request: {user_msg}\nReply with ONLY the task name."
        )
        raw = lg_orchestrator_llm.invoke([HumanMessage(content=prompt)]).content.strip().lower()
        task = raw if raw in ["lessonplan", "quiz", "email"] else "lessonplan"
    return {**state, "task_type": task}

def route_to_agent(state):
    return {"lessonplan": "lessonplanner", "quiz": "quizcontent", "email": "emaildrafter"}[state["task_type"]]

def lesson_planner_node(state):
    q = state["messages"][-1].content
    mat = retrieve_course_materials(q, page_size=6)
    resp = lg_worker_llm.invoke([HumanMessage(content=(
        "You are an expert curriculum designer at Mae Fah Luang University.\n\n"
        f"Course Materials:\n{mat}\n\nRequest:\n{q}\n\n"
        "Generate a 90-minute lesson plan: Learning Objectives, timing, methods, assessment, materials."
    ))])
    return {**state, "course_materials": mat, "final_output": resp.content,
            "messages": state["messages"] + [AIMessage(content=resp.content)]}

def quiz_content_node(state):
    q = state["messages"][-1].content
    mat = retrieve_course_materials(q, page_size=8)
    resp = lg_worker_llm.invoke([HumanMessage(content=(
        "You are a quiz specialist at Mae Fah Luang University.\n\n"
        f"Course Materials:\n{mat}\n\nTopic: {q}\n\n"
        "Generate exactly 10 MCQ questions as a JSON array only.\n"
        'Each item: {"question":"...","options":["A.","B.","C.","D."],"correct_index":0,"explanation":"..."}'
    ))])
    return {**state, "course_materials": mat, "draft_output": resp.content}

def quiz_publisher_node(state):
    resp = lg_worker_llm.invoke([HumanMessage(content=(
        f"Quiz JSON:\n{state['draft_output']}\n\n"
        "Present cleanly: number each question, A/B/C/D options, mark correct, include explanation."
    ))])
    return {**state, "final_output": resp.content,
            "messages": state["messages"] + [AIMessage(content=resp.content)]}

def email_drafter_node(state):
    q = state["messages"][-1].content
    mat = retrieve_course_materials(q, page_size=2)
    resp = lg_worker_llm.invoke([HumanMessage(content=(
        "You are a professional email assistant for Mae Fah Luang University.\n\n"
        f"Context:\n{mat}\n\nDraft email for:\n{q}\n\n"
        "Format: SUBJECT: <line>\n\nBODY:\n<body>\n\nFormal university tone."
    ))])
    return {**state, "course_materials": mat, "draft_output": resp.content}

def hitl_approval_node(state):
    return {**state, "hitl_decision": "approved"}

def email_sender_node(state):
    msg = f"Email approved and sent.\n\n{state['draft_output']}"
    return {**state, "final_output": msg,
            "messages": state["messages"] + [AIMessage(content=msg)]}

def route_after_hitl(state):
    return "emailsender"

print("LangGraph nodes defined.")

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
builder = StateGraph(TeacherState)
builder.add_node("router",        router_node)
builder.add_node("lessonplanner", lesson_planner_node)
builder.add_node("quizcontent",   quiz_content_node)
builder.add_node("quizpublisher", quiz_publisher_node)
builder.add_node("emaildrafter",  email_drafter_node)
builder.add_node("hitlapproval",  hitl_approval_node)
builder.add_node("emailsender",   email_sender_node)
builder.set_entry_point("router")
builder.add_conditional_edges("router", route_to_agent)
builder.add_edge("lessonplanner", END)
builder.add_edge("quizcontent",   "quizpublisher")
builder.add_edge("quizpublisher", END)
builder.add_edge("emaildrafter",  "hitlapproval")
builder.add_conditional_edges("hitlapproval", route_after_hitl)
builder.add_edge("emailsender",   END)
lg_graph = builder.compile(checkpointer=checkpointer)
print("LangGraph compiled:", list(lg_graph.get_graph().nodes.keys()))

## 5 — Build Google ADK System

In [ ]:
import google.adk
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import FunctionTool
from google.adk.tools.agent_tool import AgentTool
from google.adk.models.google_llm import Gemini
from google.genai import types

resilient_http = types.HttpOptions(
    retry_options=types.HttpRetryOptions(
        attempts=5, initial_delay=5.0,
        http_status_codes=[408, 429, 500, 502, 503, 504]
    )
)
adk_pro   = Gemini(model="gemini-2.5-pro",   http_options=resilient_http,
    generate_content_config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=1024)))
adk_flash = Gemini(model="gemini-2.5-flash", http_options=resilient_http,
    generate_content_config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=512)))

adk_lesson_planner = LlmAgent(
    name="lesson_planner_agent", model=adk_flash,
    description="Creates detailed lesson plans.",
    instruction="""You are an expert curriculum designer at Mae Fah Luang University.
1. Use `retrieve_course_materials` (page_size=6) to fetch relevant content.
2. Generate a 90-minute lesson plan: Learning Objectives, timing, teaching methods, assessment, materials.""",
    tools=[retrieve_course_materials]
)
quiz_content_agent = LlmAgent(
    name="quiz_content_agent", model=adk_flash,
    description="Generates quiz questions as JSON.",
    instruction="""You are a quiz specialist at Mae Fah Luang University.
1. Use `retrieve_course_materials` (page_size=8).
2. Generate exactly 10 MCQ questions as a valid JSON array.
Each item: {"question":"...","options":["A.","B.","C.","D."],"correct_index":0,"explanation":"..."}""",
    tools=[retrieve_course_materials], output_key="quiz_questions_json"
)
quiz_publisher_agent = LlmAgent(
    name="quiz_publisher_agent", model=adk_flash,
    description="Formats and presents the quiz.",
    instruction="You are a quiz publisher. Quiz JSON: {quiz_questions_json}\nPresent cleanly: numbered questions, A/B/C/D, mark correct, include explanation.",
    tools=[]
)
adk_quiz_generator = SequentialAgent(
    name="quiz_generator_agent",
    description="Generates a 10-question MCQ quiz.",
    sub_agents=[quiz_content_agent, quiz_publisher_agent]
)
email_drafter_agent = LlmAgent(
    name="email_drafter_agent", model=adk_flash,
    description="Drafts a professional email to students.",
    instruction="""You are a professional email drafting assistant for Mae Fah Luang University.
Draft a professional email based on the instructor's request.
Format EXACTLY as:
SUBJECT: <subject line>

BODY:
<email body>

Use formal university tone.""",
    output_key="email_draft"
)
def send_email_to_students(subject: str, body: str) -> dict:
    """Send email to all enrolled students.

    Args:
        subject: Email subject line.
        body: Complete email body.

    Returns:
        Confirmation dict with status and recipients.
    """
    return {"status": "sent", "subject": subject, "recipients": "all_students"}

email_sender_agent = LlmAgent(
    name="email_sender_agent", model=adk_flash,
    description="Sends approved email to students.",
    instruction="""You are an email dispatch agent.
The email draft is: {email_draft}
1. Parse the draft to extract SUBJECT and BODY.
2. Call send_email_to_students with the subject and body.
3. Report the result to the instructor.""",
    tools=[FunctionTool(func=send_email_to_students)],
    before_tool_callback=lambda tool, args, tool_context, **kw: None
)
adk_email_agent = SequentialAgent(
    name="email_agent",
    description="Drafts then sends student email.",
    sub_agents=[email_drafter_agent, email_sender_agent]
)
adk_root = LlmAgent(
    name="teacher_assistant_orchestrator", model=adk_pro,
    description="Root AI Teaching Assistant for MFU instructors.",
    instruction="""You are the AI Teaching Assistant for Mae Fah Luang University instructors.
Help with: Lesson Plans, Quizzes, Emails.
Identify task type, delegate to specialist agent, report full result.""",
    tools=[AgentTool(agent=adk_lesson_planner),
           AgentTool(agent=adk_quiz_generator),
           AgentTool(agent=adk_email_agent)]
)
print("ADK root orchestrator ready.")

## 6 — Runners

In [ ]:
import uuid, time, asyncio
from langsmith import traceable

USERID = "mfu-instructor-halluc"
ADK_APP_NAME = "teacher-assistant-adk-halluc"
adk_session_service = InMemorySessionService()
adk_runner = Runner(agent=adk_root, app_name=ADK_APP_NAME, session_service=adk_session_service)

@traceable(name="lg_halluc_run", run_type="chain",
           metadata={"framework": "LangGraph", "notebook": "05_hallucination"})
def lg_run(user_input: str, run_key: str):
    set_run_key(run_key)
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id},
              "metadata": {"run_key": run_key, "framework": "LangGraph"}}
    init_state = {"messages": [HumanMessage(content=user_input)],
                  "task_type": "", "course_materials": "",
                  "draft_output": "", "final_output": "", "hitl_decision": ""}
    start = time.time()
    result = lg_graph.invoke(init_state, config)
    latency = round(time.time() - start, 2)
    contexts = list(CONTEXT_STORE.get(run_key, []))
    print(f"  [LangGraph] {latency}s | {len(contexts)} context chunks")
    return result.get("final_output", ""), latency, contexts

@traceable(name="adk_halluc_run", run_type="chain",
           metadata={"framework": "Google ADK", "notebook": "05_hallucination"})
async def adk_run(user_input: str, run_key: str):
    set_run_key(run_key)
    session_id = str(uuid.uuid4())
    await adk_session_service.create_session(
        app_name=ADK_APP_NAME, user_id=USERID, session_id=session_id)
    message = types.Content(role="user", parts=[types.Part(text=user_input)])
    start = time.time()
    final_response = ""
    async for event in adk_runner.run_async(
        user_id=USERID, session_id=session_id, new_message=message):
        if event.is_final_response() and event.content:
            final_response = event.content.parts[0].text.strip()
    latency = round(time.time() - start, 2)
    contexts = list(CONTEXT_STORE.get(run_key, []))
    print(f"  [ADK] {latency}s | {len(contexts)} context chunks")
    return final_response, latency, contexts

print("Runners ready.")

## 7 — Run Scenarios + Capture Contexts

In [ ]:
SCENARIOS = {
    "lesson_plan": "Create a 90-minute lesson plan for week 1 on Software Testing for second-year Software Engineering students. Align it with the course materials.",
    "quiz":        "Generate 10 multiple-choice questions on Software Testing from the course materials.",
    "email":       "Draft and send an email to all students reminding them that the Software Testing quiz covering Unit Testing and Black Box Testing is next Monday at 9am. Include what topics to study.",
}

N_RUNS = 1  # Increase to 3 for statistical averages

# eval_data[scenario][framework] = list of {question, answer, contexts, latency}
eval_data = {}

for scenario_key, prompt in SCENARIOS.items():
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_key}")
    print("="*60)
    eval_data[scenario_key] = {"lg": [], "adk": []}

    for run_i in range(N_RUNS):
        print(f"\n  Run {run_i+1}/{N_RUNS}")

        print("  [LangGraph] running...")
        try:
            rk = f"{scenario_key}_lg_r{run_i}"
            ans, lat, ctx = lg_run(user_input=prompt, run_key=rk)
            eval_data[scenario_key]["lg"].append(
                {"question": prompt, "answer": ans, "contexts": ctx, "latency": lat, "error": None})
        except Exception as e:
            eval_data[scenario_key]["lg"].append(
                {"question": prompt, "answer": "", "contexts": [], "latency": 0, "error": str(e)})
            print(f"  ERROR: {e}")

        await asyncio.sleep(15)

        print("  [ADK] running...")
        try:
            rk = f"{scenario_key}_adk_r{run_i}"
            ans, lat, ctx = await adk_run(user_input=prompt, run_key=rk)
            eval_data[scenario_key]["adk"].append(
                {"question": prompt, "answer": ans, "contexts": ctx, "latency": lat, "error": None})
        except Exception as e:
            eval_data[scenario_key]["adk"].append(
                {"question": prompt, "answer": "", "contexts": [], "latency": 0, "error": str(e)})
            print(f"  ERROR: {e}")

        if run_i < N_RUNS - 1:
            await asyncio.sleep(30)

print("\n✓ All scenarios completed.")

# Sanity check
for sc, fw_data in eval_data.items():
    for fw, runs in fw_data.items():
        for r in runs:
            print(f"{sc:12} | {fw:3} | ctx={len(r['contexts'])} | ans={len(r['answer'])} chars | err={r['error']}")

## 8 — DeepEval Hallucination Measurement

> **HallucinationMetric** extracts atomic claims from the output
> and verifies each against the provided `context` list.
> Score = fraction of claims **NOT supported** by context.
> `0.0` = no hallucination, `1.0` = fully hallucinated.

We use a **Gemini wrapper** so no OpenAI key is needed.

In [ ]:
from deepeval.metrics import HallucinationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM
from langchain_google_genai import ChatGoogleGenerativeAI

# ── Gemini wrapper for DeepEval ──────────────────────────────────
class GeminiDeepEvalLLM(DeepEvalBaseLLM):
    """Wraps Gemini Flash via LangChain for use inside DeepEval metrics."""

    def __init__(self):
        self._llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash", vertexai=True, project=PROJECT_ID,
            location="us-central1", max_output_tokens=2048,
        )

    def load_model(self):
        return self._llm

    def generate(self, prompt: str) -> str:
        return self._llm.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        result = await self._llm.ainvoke(prompt)
        return result.content

    def get_model_name(self) -> str:
        return "gemini-2.5-flash-vertexai"

gemini_judge = GeminiDeepEvalLLM()
print("DeepEval Gemini judge ready:", gemini_judge.get_model_name())

In [ ]:
import pandas as pd

HALLUC_THRESHOLD = 0.5  # Score >= 0.5 = classified as hallucinating

halluc_metric = HallucinationMetric(
    threshold=HALLUC_THRESHOLD,
    model=gemini_judge,
    include_reason=True,
)

deepeval_rows = []

for scenario_key, prompt in SCENARIOS.items():
    print(f"\n{'='*60}")
    print(f"DeepEval: {scenario_key}")
    print("="*60)

    for fw_key, fw_label in [("lg", "LangGraph"), ("adk", "Google ADK")]:
        for run_i, run_data in enumerate(eval_data[scenario_key][fw_key]):
            if run_data["error"] or not run_data["answer"]:
                deepeval_rows.append({
                    "framework": fw_label, "scenario": scenario_key, "run": run_i+1,
                    "hallucination_score": 1.0, "is_hallucinating": True,
                    "reason": "Empty or errored response.",
                    "n_contexts": 0, "latency": run_data["latency"]
                })
                continue

            ctx = run_data["contexts"]
            if not ctx:
                ctx = ["No context retrieved from datastore."]

            test_case = LLMTestCase(
                input=prompt,
                actual_output=run_data["answer"],
                context=ctx,
            )

            print(f"  [{fw_label}] run {run_i+1}...", end=" ")
            try:
                halluc_metric.measure(test_case)
                score  = halluc_metric.score          # 0.0 – 1.0
                passed = halluc_metric.is_successful() # True = below threshold (good)
                reason = halluc_metric.reason or ""
                print(f"score={score:.3f} | {'✓ OK' if passed else '✗ HALLUC'} | {reason[:80]}")
            except Exception as e:
                score, passed, reason = 1.0, False, str(e)
                print(f"ERROR: {e}")

            deepeval_rows.append({
                "framework": fw_label, "scenario": scenario_key, "run": run_i+1,
                "hallucination_score": score,
                "is_hallucinating": not passed,
                "reason": reason,
                "n_contexts": len(ctx),
                "latency": run_data["latency"]
            })

df_deepeval = pd.DataFrame(deepeval_rows)
print("\n✓ DeepEval scoring complete.")
print(df_deepeval[["framework","scenario","hallucination_score","is_hallucinating","n_contexts"]].to_string())

## 9 — Load Prior RAGAS + Judge Scores (Optional)

> If you ran NB03 and NB04, upload `judge_results.csv` and `ragas_results.csv`
> to combine all three evaluation lenses into the composite score.
> Skip this section and use the standalone composite in Section 10 if files are unavailable.

In [ ]:
import os

# ── Attempt to load prior results ────────────────────────────────
JUDGE_CSV = "judge_results.csv"
RAGAS_CSV = "ragas_results.csv"

df_judge = None
df_ragas = None

if os.path.exists(JUDGE_CSV):
    df_judge = pd.read_csv(JUDGE_CSV)
    print(f"Loaded judge results: {len(df_judge)} rows")
    print("  Columns:", df_judge.columns.tolist())
else:
    print(f"'{JUDGE_CSV}' not found. Upload from NB03 output or skip.")

if os.path.exists(RAGAS_CSV):
    df_ragas = pd.read_csv(RAGAS_CSV)
    print(f"Loaded RAGAS results: {len(df_ragas)} rows")
    print("  Columns:", df_ragas.columns.tolist())
else:
    print(f"'{RAGAS_CSV}' not found. Upload from NB04 output or skip.")

# ── Fallback: use current run's NaN placeholders ──────────────────
# These will be filled in Section 10 with current-run scores
print("\nProceeding — composite will use available data.")

## 10 — Composite Hallucination Rate

$$\text{Composite Hallucination Rate} = \frac{(1 - \text{Faithfulness}) + \text{DeepEval Score} + (1 - \frac{\text{Accuracy}}{5})}{3} \times 100$$

Each component is normalized to `[0, 1]` before averaging:
- `0` = no hallucination
- `1` = fully hallucinated

In [ ]:
import numpy as np

SCENARIOS_LIST = list(SCENARIOS.keys())
FRAMEWORKS = ["LangGraph", "Google ADK"]

def get_score(df, framework, scenario, col, agg="mean"):
    """Safely extract a score from a DataFrame."""
    if df is None:
        return np.nan
    sub = df[(df["framework"] == framework) & (df["scenario"] == scenario)]
    if len(sub) == 0 or col not in sub.columns:
        return np.nan
    return sub[col].mean() if agg == "mean" else sub[col].iloc[0]

composite_rows = []

for fw in FRAMEWORKS:
    for sc in SCENARIOS_LIST:
        # DeepEval hallucination score (always available from this notebook)
        sub_de = df_deepeval[(df_deepeval["framework"] == fw) & (df_deepeval["scenario"] == sc)]
        deepeval_score = sub_de["hallucination_score"].mean() if len(sub_de) > 0 else np.nan

        # RAGAS Faithfulness (from NB04 CSV or NaN)
        faithfulness = get_score(df_ragas, fw, sc, "faithfulness")
        ragas_component = (1 - faithfulness) if not np.isnan(faithfulness) else np.nan

        # LLM Judge Accuracy (from NB03 CSV or NaN)
        accuracy = get_score(df_judge, fw, sc, "accuracy")
        judge_component = (1 - accuracy / 5.0) if not np.isnan(accuracy) else np.nan

        # Composite: mean of available components
        components = [c for c in [ragas_component, deepeval_score, judge_component] if not np.isnan(c)]
        composite = (sum(components) / len(components)) * 100 if components else np.nan

        avg_lat = df_deepeval[(df_deepeval["framework"] == fw) & (df_deepeval["scenario"] == sc)]["latency"].mean()

        composite_rows.append({
            "framework":         fw,
            "scenario":          sc,
            "deepeval_score":    round(deepeval_score, 3) if not np.isnan(deepeval_score) else None,
            "ragas_faithfulness": round(faithfulness, 3) if not np.isnan(faithfulness) else None,
            "judge_accuracy":    round(accuracy, 3) if not np.isnan(accuracy) else None,
            "composite_halluc_pct": round(composite, 1) if not np.isnan(composite) else None,
            "avg_latency":       round(avg_lat, 2)
        })

df_composite = pd.DataFrame(composite_rows)

print("=" * 70)
print("  Composite Hallucination Rate (%) — lower is better")
print("=" * 70)
print(df_composite.to_string(index=False))

print("\n" + "=" * 70)
print("  Overall per Framework")
print("=" * 70)
overall = df_composite.groupby("framework")[["deepeval_score","composite_halluc_pct","avg_latency"]].mean().round(2)
print(overall.to_string())

## 11 — Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

COLORS = {"LangGraph": "#4C72B0", "Google ADK": "#DD8452"}
x = np.arange(len(SCENARIOS_LIST))
width = 0.35

# ── Chart 1: DeepEval hallucination score per scenario ────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_title("DeepEval Hallucination Score per Scenario\n(0.0 = no hallucination, 1.0 = fully hallucinated)",
             fontsize=12, fontweight="bold")
ax.axhline(HALLUC_THRESHOLD, color="red", linestyle="--", linewidth=1.2, label=f"Threshold ({HALLUC_THRESHOLD})")

for i, fw in enumerate(FRAMEWORKS):
    vals = [
        df_composite[(df_composite["framework"] == fw) & (df_composite["scenario"] == sc)]["deepeval_score"].values[0]
        for sc in SCENARIOS_LIST
    ]
    bars = ax.bar(x + i*width - width/2, vals, width, label=fw, color=COLORS[fw], alpha=0.85)
    for bar, v in zip(bars, vals):
        if v is not None:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([s.replace("_", " ").title() for s in SCENARIOS_LIST], fontsize=10)
ax.set_ylabel("Hallucination Score (0–1)", fontsize=10)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("hallucination_bar.png", dpi=150)
plt.show()
print("Chart 1 saved: hallucination_bar.png")

In [ ]:
# ── Chart 2: Composite hallucination rate heatmap ─────────────────
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_title("Composite Hallucination Rate (%) — lower is better",
             fontsize=12, fontweight="bold")

matrix = []
for fw in FRAMEWORKS:
    row = [
        df_composite[(df_composite["framework"] == fw) & (df_composite["scenario"] == sc)]["composite_halluc_pct"].values[0]
        for sc in SCENARIOS_LIST
    ]
    matrix.append(row)

matrix = np.array([[v if v is not None else np.nan for v in row] for row in matrix], dtype=float)

# Reverse colormap: high % = red, low % = green
im = ax.imshow(matrix, vmin=0, vmax=100, cmap="RdYlGn_r", aspect="auto")
ax.set_xticks(range(len(SCENARIOS_LIST)))
ax.set_xticklabels([s.replace("_", " ").title() for s in SCENARIOS_LIST], fontsize=10)
ax.set_yticks(range(len(FRAMEWORKS)))
ax.set_yticklabels(FRAMEWORKS, fontsize=10)

for i in range(len(FRAMEWORKS)):
    for j in range(len(SCENARIOS_LIST)):
        val = matrix[i, j]
        if not np.isnan(val):
            txt_color = "white" if val > 60 or val < 20 else "black"
            ax.text(j, i, f"{val:.1f}%", ha="center", va="center",
                    fontsize=11, fontweight="bold", color=txt_color)

plt.colorbar(im, ax=ax, fraction=0.03, label="Hallucination Rate (%)")
plt.tight_layout()
plt.savefig("hallucination_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart 2 saved: hallucination_heatmap.png")

In [ ]:
# ── Chart 3: Three-component breakdown stacked bar ────────────────
fig, axes = plt.subplots(1, len(FRAMEWORKS), figsize=(12, 5), sharey=True)
fig.suptitle("Hallucination Components per Framework\n(stacked: RAGAS + DeepEval + Judge)",
             fontsize=12, fontweight="bold")

component_colors = ["#e07b54", "#c0392b", "#922b21"]
component_labels = ["1 - Faithfulness", "DeepEval Score", "1 - Accuracy/5"]

for ax, fw in zip(axes, FRAMEWORKS):
    sub = df_composite[df_composite["framework"] == fw]
    sc_labels = [s.replace("_", "\n").title() for s in SCENARIOS_LIST]
    bottoms = np.zeros(len(SCENARIOS_LIST))

    components_data = [
        [((1 - sub[sub["scenario"]==sc]["ragas_faithfulness"].values[0])
          if sub[sub["scenario"]==sc]["ragas_faithfulness"].values[0] is not None else 0)
         for sc in SCENARIOS_LIST],
        [(sub[sub["scenario"]==sc]["deepeval_score"].values[0] or 0) for sc in SCENARIOS_LIST],
        [((1 - sub[sub["scenario"]==sc]["judge_accuracy"].values[0] / 5.0)
          if sub[sub["scenario"]==sc]["judge_accuracy"].values[0] is not None else 0)
         for sc in SCENARIOS_LIST],
    ]

    for comp_vals, col, lbl in zip(components_data, component_colors, component_labels):
        ax.bar(range(len(SCENARIOS_LIST)), comp_vals, bottom=bottoms,
               color=col, label=lbl, alpha=0.85)
        bottoms += np.array(comp_vals)

    ax.set_xticks(range(len(SCENARIOS_LIST)))
    ax.set_xticklabels(sc_labels, fontsize=9)
    ax.set_title(fw, fontsize=11, fontweight="bold")
    ax.set_ylim(0, 3.3)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

axes[0].set_ylabel("Hallucination Component Sum (0–3)", fontsize=10)
handles = [mpatches.Patch(color=c, label=l) for c, l in zip(component_colors, component_labels)]
fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.savefig("hallucination_components.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart 3 saved: hallucination_components.png")

## 12 — Export Results

In [ ]:
from datetime import datetime, timezone
from google.colab import files

# Save DeepEval raw scores
deepeval_csv = "hallucination_results.csv"
df_deepeval.to_csv(deepeval_csv, index=False)
print(f"Saved: {deepeval_csv} ({len(df_deepeval)} rows)")

# Save composite table
composite_csv = "hallucination_composite.csv"
df_composite.to_csv(composite_csv, index=False)
print(f"Saved: {composite_csv}")

# Save markdown report
report_md = "hallucination_report.md"
with open(report_md, "w") as f:
    f.write("# Hallucination Rate Analysis Report\n")
    f.write(f"Generated: {datetime.now(timezone.utc).isoformat()}\n\n")
    f.write("## Formula\n\n")
    f.write("Composite Hallucination Rate = ((1 - Faithfulness) + DeepEval Score + (1 - Accuracy/5)) / 3 × 100\n\n")
    f.write("## Composite Results\n\n")
    f.write("```\n" + df_composite.to_string(index=False) + "\n```\n\n")
    f.write("## Overall per Framework\n\n")
    f.write("```\n" + overall.to_string() + "\n```\n\n")
    f.write("## DeepEval Claim-Level Reasoning\n\n")
    for _, row in df_deepeval.iterrows():
        f.write(f"### {row['framework']} | {row['scenario']} (Run {row['run']})\n")
        f.write(f"- Score: {row['hallucination_score']:.3f} | "
                f"Hallucinating: {row['is_hallucinating']} | "
                f"Contexts: {row['n_contexts']}\n")
        f.write(f"- Reason: {str(row['reason'])[:300]}\n\n")

print(f"Saved: {report_md}")

files.download(deepeval_csv)
files.download(composite_csv)
files.download(report_md)
print("\nDownloads triggered.")